In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt


I0000 00:00:1780245761.046177   23469 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1780245761.050173   23469 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1780245761.573034   23469 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1780245763.664223   23469 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

In [2]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models

# EfficientNetB0 default input size is 224x224
IMG_SIZE = 224

# Load the base model without the top classification layers
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))

# Freeze the pretrained backbone for transfer learning
base_model.trainable = False
print('Base model trainable:', base_model.trainable)
print('Trainable layers in base:', sum(int(l.trainable) for l in base_model.layers), '/', len(base_model.layers))


E0000 00:00:1780245764.756448   23469 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Base model trainable: False
Trainable layers in base: 0 / 238


In [3]:
df = pd.read_csv('dataset/image_data.csv')

# Drop rows whose image file is missing (prevents tf.io.read_file crashes)
from pathlib import Path
_img_dir = Path('dataset/images')
_exists = df['filename'].astype(str).map(lambda n: (_img_dir / n).exists())
_missing = (~_exists).sum()
if _missing:
    print(f"Warning: dropping {_missing} rows with missing files")
    df = df[_exists].reset_index(drop=True)

# Ensure expected dtypes
df['vehicle_type'] = df['vehicle_type'].astype(str)
df['cope_cage'] = df['cope_cage'].astype(int)
df['destroyed'] = df['destroyed'].astype(int)

print('Rows after cleanup:', len(df))
# df

Rows after cleanup: 130


In [4]:
vehicle_list = sorted(df['vehicle_type'].unique())
num_vehicle_classes = len(vehicle_list)

In [5]:

# Map vehicle string labels to sequential integers (e.g., 'btr' -> 0, 'tank' -> 1)
label_to_index = {name: i for i, name in enumerate(vehicle_list)}
df['vehicle_id'] = df['vehicle_type'].map(label_to_index)

In [6]:
print(f"Detected {num_vehicle_classes} unique vehicle classes: {vehicle_list}")


Detected 2 unique vehicle classes: ['btr', 'tank']


In [7]:
# --- Build + compile multi-head model ---

# GlobalAveragePooling is typically better than Flatten for transfer learning
x = layers.GlobalAveragePooling2D()(base_model.output)
x = layers.Dense(128, activation='relu')(x)

type_head = layers.Dense(num_vehicle_classes, activation='softmax', name='type_output_layer')(x)
cage_head = layers.Dense(1, activation='sigmoid', name='cage_output_layer')(x)
status_head = layers.Dense(1, activation='sigmoid', name='status_output_layer')(x)

model = models.Model(inputs=base_model.input, outputs=[type_head, cage_head, status_head])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss={
        'type_output_layer': tf.keras.losses.SparseCategoricalCrossentropy(),
        'cage_output_layer': tf.keras.losses.BinaryCrossentropy(),
        'status_output_layer': tf.keras.losses.BinaryCrossentropy(),
    },
    metrics={
        'type_output_layer': 'accuracy',
        'cage_output_layer': 'accuracy',
        'status_output_layer': 'accuracy',
    },
)

print('Built + compiled model with num_vehicle_classes =', num_vehicle_classes)
print('Backbone:', base_model.name, 'Input:', model.input_shape)


Built + compiled model with num_vehicle_classes = 2
Backbone: efficientnetb0 Input: (None, 224, 224, 3)


In [ ]:
def process_path(filename, vehicle_id, cope_cage, destroyed):
    """Load + decode image, resize to IMG_SIZE, and return labels for multi-head training."""
    # Ensure filename is a scalar string tensor (some pipelines can introduce extra dims)
    filename = tf.reshape(tf.squeeze(filename), [])
    tf.debugging.assert_rank(filename, 0, message='filename must be scalar')
    img_path = tf.strings.join(['dataset/images/', filename])
    tf.debugging.assert_rank(img_path, 0, message='img_path must be scalar')

    # Read bytes
    img_bytes = tf.io.read_file(img_path)
    tf.debugging.assert_rank(img_bytes, 0, message='img_bytes must be scalar')

    # Decode to a fixed 3-channel RGB tensor (handles JPEG/PNG/WEBP, etc.)
    img = tf.image.decode_image(img_bytes, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])

    # Resize to the backbone input size
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))

    # EfficientNetB0 (with built-in preprocessing) expects float inputs in [0, 255]
    img = tf.cast(img, tf.float32)

    return img, {
        'type_output_layer': tf.cast(vehicle_id, tf.int32),
        'cage_output_layer': tf.cast(cope_cage, tf.float32),
        'status_output_layer': tf.cast(destroyed, tf.float32),
    }


In [9]:
# --- Stratified train/validation split (combined key: vehicle_type|cope_cage|destroyed) ---

SEED = 42
VAL_FRAC = 0.20

rng = np.random.RandomState(SEED)

strat_key = (
    df["vehicle_type"].astype(str)
    + "|c" + df["cope_cage"].astype(int).astype(str)
    + "|d" + df["destroyed"].astype(int).astype(str)
)

print("Stratum counts (vehicle_type|cope_cage|destroyed):")
display(strat_key.value_counts())

train_indices = []
val_indices = []

for key, group in df.groupby(strat_key):
    idx = group.index.to_numpy().copy()
    rng.shuffle(idx)
    n = len(idx)

    n_val = int(round(n * VAL_FRAC))
    if n >= 2:
        n_val = min(max(1, n_val), n - 1)
    else:
        # Can't split a singleton stratum; keep it in train.
        n_val = 0

    val_indices.extend(idx[:n_val].tolist())
    train_indices.extend(idx[n_val:].tolist())

train_df = df.loc[train_indices].sample(frac=1, random_state=SEED).reset_index(drop=True)
val_df = df.loc[val_indices].sample(frac=1, random_state=SEED).reset_index(drop=True)

if len(val_df) == 0 or len(train_df) == 0:
    raise ValueError(f"Invalid split produced. train={len(train_df)} val={len(val_df)}")

print(f"\nSplit sizes: train={len(train_df)}  val={len(val_df)}  (val_frac={len(val_df)/len(df):.3f})")
print("Validation combined-key counts:")
display((val_df["vehicle_type"].astype(str)
         + "|c" + val_df["cope_cage"].astype(int).astype(str)
         + "|d" + val_df["destroyed"].astype(int).astype(str)).value_counts())

# --- Train Generator ---
train_dataset = tf.data.Dataset.from_tensor_slices((
    train_df['filename'].values,
    train_df['vehicle_id'].values,
    train_df['cope_cage'].values,
    train_df['destroyed'].values,
))
train_generator = train_dataset.map(process_path, num_parallel_calls=tf.data.AUTOTUNE).batch(32).prefetch(tf.data.AUTOTUNE)

# --- Validation Generator ---
val_dataset = tf.data.Dataset.from_tensor_slices((
    val_df['filename'].values,
    val_df['vehicle_id'].values,
    val_df['cope_cage'].values,
    val_df['destroyed'].values,
))
val_generator = val_dataset.map(process_path, num_parallel_calls=tf.data.AUTOTUNE).batch(32).prefetch(tf.data.AUTOTUNE)


Stratum counts (vehicle_type|cope_cage|destroyed):


btr|c0|d0     31
tank|c1|d0    24
tank|c0|d0    23
btr|c0|d1     21
tank|c0|d1    18
btr|c1|d0     10
tank|c1|d1     2
btr|c1|d1      1
Name: count, dtype: int64


Split sizes: train=103  val=27  (val_frac=0.208)
Validation combined-key counts:


btr|c0|d0     6
tank|c0|d0    5
tank|c1|d0    5
btr|c0|d1     4
tank|c0|d1    4
btr|c1|d0     2
tank|c1|d1    1
Name: count, dtype: int64

In [10]:
# --- Inspect val_dataset / val_generator ---

# 1) Inspect the underlying validation DataFrame

display(val_df[["filename", "vehicle_type", "vehicle_id", "cope_cage", "destroyed"]])
print("VALUE COUNTS:")
print(val_df["vehicle_type"].value_counts())
print(val_df["cope_cage"].value_counts())
print(val_df["destroyed"].value_counts())

# 2) Inspect the tf.data signatures

print("val_dataset.element_spec:")
print(val_dataset.element_spec)

print("\nval_generator.element_spec (after process_path + batching):")
print(val_generator.element_spec)

# 3) Preview a few raw (unprocessed) items from val_dataset
preview = []
for filename, vehicle_id, cope_cage, destroyed in val_dataset.take(10):
    fn = filename.numpy()
    if isinstance(fn, (bytes, bytearray)):
        fn = fn.decode("utf-8")
    vid = int(vehicle_id.numpy())
    preview.append({
        "filename": str(fn),
        "vehicle_id": vid,
        "vehicle_type": vehicle_list[vid] if 0 <= vid < len(vehicle_list) else None,
        "cope_cage": int(cope_cage.numpy()),
        "destroyed": int(destroyed.numpy()),
    })

display(pd.DataFrame(preview))

# 4) Preview a processed batch from val_generator
images, labels = next(iter(val_generator.take(1)))
print("\nProcessed batch shapes:")
print("images:", images.shape)
print("labels keys:", list(labels.keys()))
for k, v in labels.items():
    print(f"{k}: {v.shape}")


,filename,vehicle_type,vehicle_id,cope_cage,destroyed
0,btr42.jpeg,btr,0,0,1
1,tank49.jpeg,tank,1,0,0
2,btr45.jpeg,btr,0,0,1
3,tank60.jpeg,tank,1,1,0
4,btr59.jpeg,btr,0,0,0
5,btr22.jpeg,btr,0,1,0
6,tank47.jpeg,tank,1,0,0
7,tank20.jpeg,tank,1,0,1
8,tank23.jpeg,tank,1,0,0
9,tank5.jpeg,tank,1,1,0


VALUE COUNTS:
vehicle_type
tank    15
btr     12
Name: count, dtype: int64
cope_cage
0    19
1     8
Name: count, dtype: int64
destroyed
0    18
1     9
Name: count, dtype: int64
val_dataset.element_spec:
(TensorSpec(shape=(), dtype=tf.string, name=None), TensorSpec(shape=(), dtype=tf.int64, name=None), TensorSpec(shape=(), dtype=tf.int64, name=None), TensorSpec(shape=(), dtype=tf.int64, name=None))

val_generator.element_spec (after process_path + batching):
(TensorSpec(shape=(None, 150, 150, None), dtype=tf.float32, name=None), {'type_output_layer': TensorSpec(shape=(None,), dtype=tf.int64, name=None), 'cage_output_layer': TensorSpec(shape=(None,), dtype=tf.int64, name=None), 'status_output_layer': TensorSpec(shape=(None,), dtype=tf.int64, name=None)})


,filename,vehicle_id,vehicle_type,cope_cage,destroyed
0,btr42.jpeg,0,btr,0,1
1,tank49.jpeg,1,tank,0,0
2,btr45.jpeg,0,btr,0,1
3,tank60.jpeg,1,tank,1,0
4,btr59.jpeg,0,btr,0,0
5,btr22.jpeg,0,btr,1,0
6,tank47.jpeg,1,tank,0,0
7,tank20.jpeg,1,tank,0,1
8,tank23.jpeg,1,tank,0,0
9,tank5.jpeg,1,tank,1,0



Processed batch shapes:
images: (27, 150, 150, 3)
labels keys: ['type_output_layer', 'cage_output_layer', 'status_output_layer']
type_output_layer: (27,)
cage_output_layer: (27,)
status_output_layer: (27,)


In [11]:
# --- Train model on stratified holdout split ---

EPOCHS = 20

print("Starting training pipeline...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
)
print("Training complete!")


Starting training pipeline...
Epoch 1/20


ValueError: Input 0 with name 'input_layer' of layer 'functional' is incompatible with the layer: expected shape=(None, 224, 224, 3), found shape=(None, 150, 150, None)

In [ ]:
# --- Visualisations: total + per-head loss curves (train vs val) ---

hist = history.history
epochs = range(1, len(hist["loss"]) + 1)

def plot_train_vs_val(key: str, title: str = None):
    train_key = key
    val_key = f"val_{key}"
    if train_key not in hist or val_key not in hist:
        print(f"Missing keys: {train_key!r} or {val_key!r}. Available keys: {list(hist.keys())}")
        return

    plt.figure(figsize=(7, 4))
    plt.plot(epochs, hist[train_key], label="train")
    plt.plot(epochs, hist[val_key], label="val")
    plt.xlabel("Epoch")
    plt.ylabel(key)
    plt.title(title or f"Train vs Val: {key}")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

# Overall combined loss (sum of the 3 head losses)
plot_train_vs_val("loss", "Overall Loss (Train vs Val)")

# Per-head losses
plot_train_vs_val("type_output_layer_loss", "Vehicle Type Loss (Train vs Val)")
plot_train_vs_val("cage_output_layer_loss", "Cope Cage Loss (Train vs Val)")
plot_train_vs_val("status_output_layer_loss", "Destroyed Loss (Train vs Val)")


In [ ]:
# --- Confusion matrix for vehicle type (type_output_layer) on the validation set ---

# Collect true labels
type_true = []
for _, y in val_generator:
    type_true.append(y['type_output_layer'].numpy())
type_true = np.concatenate(type_true, axis=0)

# Predict
preds = model.predict(val_generator, verbose=0)
if isinstance(preds, (list, tuple)):
    type_probs = preds[0]
elif isinstance(preds, dict):
    type_probs = preds['type_output_layer']
else:
    raise TypeError(f'Unexpected predict output type: {type(preds)}')

type_pred = np.argmax(type_probs, axis=1)
cm = tf.math.confusion_matrix(type_true, type_pred, num_classes=num_vehicle_classes).numpy()

fig, ax = plt.subplots(figsize=(6, 6))
im = ax.imshow(cm, cmap='Blues')
ax.figure.colorbar(im, ax=ax)

ax.set(
    xticks=np.arange(num_vehicle_classes),
    yticks=np.arange(num_vehicle_classes),
    xticklabels=vehicle_list,
    yticklabels=vehicle_list,
    xlabel='Predicted label',
    ylabel='True label',
    title='Confusion Matrix (Vehicle Type)'
)
plt.setp(ax.get_xticklabels(), rotation=45, ha='right', rotation_mode='anchor')

# Write counts in each cell
threshold = cm.max() / 2.0 if cm.max() else 0.0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(
            j,
            i,
            int(cm[i, j]),
            ha='center',
            va='center',
            color='white' if cm[i, j] > threshold else 'black',
        )

fig.tight_layout()
plt.show()


In [ ]:
# --- Metrics helpers (accuracy / precision / recall / F1) ---

def _safe_div(numer: float, denom: float) -> float:
    return float(numer) / float(denom) if denom else 0.0

def binary_metrics_from_probs(y_true, y_prob, threshold: float = 0.5):
    """Compute binary accuracy/precision/recall/F1 given true labels and predicted probabilities."""
    y_true = np.asarray(y_true).astype(int).reshape(-1)
    y_prob = np.asarray(y_prob).reshape(-1)
    y_pred = (y_prob >= threshold).astype(int)

    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))

    accuracy = _safe_div(tp + tn, tp + tn + fp + fn)
    precision = _safe_div(tp, tp + fp)
    recall = _safe_div(tp, tp + fn)
    f1 = _safe_div(2 * precision * recall, precision + recall) if (precision + recall) else 0.0

    return {
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "threshold": float(threshold),
        "support": int(tp + tn + fp + fn),
    }

def multiclass_metrics_from_cm(cm: np.ndarray, class_names):
    """Per-class precision/recall/F1 + overall accuracy from a confusion matrix."""
    cm = np.asarray(cm, dtype=np.int64)
    class_names = list(class_names)

    total = int(cm.sum())
    correct = int(np.trace(cm))
    overall_acc = _safe_div(correct, total)

    per_rows = []
    supports = cm.sum(axis=1)
    preds = cm.sum(axis=0)

    for i, name in enumerate(class_names):
        tp = int(cm[i, i])
        fp = int(preds[i] - tp)
        fn = int(supports[i] - tp)
        precision = _safe_div(tp, tp + fp)
        recall = _safe_div(tp, tp + fn)
        f1 = _safe_div(2 * precision * recall, precision + recall) if (precision + recall) else 0.0
        per_rows.append({"class": name, "support": int(supports[i]), "precision": precision, "recall": recall, "f1": f1})

    per_df = pd.DataFrame(per_rows)
    macro = per_df[["precision", "recall", "f1"]].mean(numeric_only=True)
    weights = per_df["support"].to_numpy()
    weighted = (per_df[["precision", "recall", "f1"]].to_numpy() * weights[:, None]).sum(axis=0) / max(1, weights.sum())

    summary_df = pd.DataFrame(
        [
            {"avg": "overall", "accuracy": overall_acc, "precision": np.nan, "recall": np.nan, "f1": np.nan, "support": total},
            {"avg": "macro", "accuracy": np.nan, "precision": float(macro["precision"]), "recall": float(macro["recall"]), "f1": float(macro["f1"]), "support": total},
            {"avg": "weighted", "accuracy": np.nan, "precision": float(weighted[0]), "recall": float(weighted[1]), "f1": float(weighted[2]), "support": total},
        ]
    )

    return per_df, summary_df


In [ ]:
# --- Compute metrics on the validation set for each head ---

# Collect true labels from the validation generator
_type_true = []
_cage_true = []
_status_true = []

for _, y in val_generator:
    _type_true.append(y["type_output_layer"].numpy())
    _cage_true.append(y["cage_output_layer"].numpy())
    _status_true.append(y["status_output_layer"].numpy())

_type_true = np.concatenate(_type_true, axis=0).reshape(-1)
_cage_true = np.concatenate(_cage_true, axis=0).reshape(-1)
_status_true = np.concatenate(_status_true, axis=0).reshape(-1)

# Predict probabilities for all heads
_preds = model.predict(val_generator, verbose=0)
if isinstance(_preds, (list, tuple)):
    _type_probs, _cage_prob, _status_prob = _preds
elif isinstance(_preds, dict):
    _type_probs = _preds["type_output_layer"]
    _cage_prob = _preds["cage_output_layer"]
    _status_prob = _preds["status_output_layer"]
else:
    raise TypeError(f"Unexpected predict output type: {type(_preds)}")

_type_probs = np.asarray(_type_probs)
_cage_prob = np.asarray(_cage_prob).reshape(-1)
_status_prob = np.asarray(_status_prob).reshape(-1)

# 1) Vehicle type head (multi-class)
_type_pred = np.argmax(_type_probs, axis=1).reshape(-1)
_cm_type = tf.math.confusion_matrix(_type_true, _type_pred, num_classes=num_vehicle_classes).numpy()

per_class_df, summary_df = multiclass_metrics_from_cm(_cm_type, vehicle_list)
print("Vehicle type head metrics (from confusion matrix)")
display(per_class_df)
display(summary_df)

# 2) Binary heads (cope cage + destroyed)
THRESHOLD = 0.5
rows = []
rows.append({"head": "cage_output_layer", **binary_metrics_from_probs(_cage_true, _cage_prob, threshold=THRESHOLD)})
rows.append({"head": "status_output_layer", **binary_metrics_from_probs(_status_true, _status_prob, threshold=THRESHOLD)})

bin_df = pd.DataFrame(rows)
print(f"Binary heads metrics (threshold={THRESHOLD})")
display(bin_df[["head", "accuracy", "precision", "recall", "f1", "tp", "tn", "fp", "fn", "support"]])
